# M3Care imputation-quality evaluation — Hospitalisation

This notebook evaluates artificially masked, originally observed modality embeddings across selected **current-window** and **dual-window** configurations for the hospitalisation pathway. **Model A = Without CL** and **Model B = With CL**. Each model's imputed representations (GCN outputs) are compared against the original target embeddings produced by **its own encoder**, not with embeddings from the other model.

The notebook reports **three main evaluation figures**, combining the window/modality configurations selected in `TEST_CONFIGURATIONS`. “Original” denotes the target embedding before masking. “Learned target” denotes the learned cross-modal target network's output. These are representation-level diagnostics, not image reconstruction or downstream clinical performance.

| Figure | Display | Main question |
|---|---|---|
| 1 | Absolute GCN imputation quality | How closely does each model reconstruct each selected modality? |
| 2 | Paired CL effect | What is the mean within-observation change (With CL − Without CL)? |
| 3 | Learned-target alignment diagnostic | How do Learned target → original, GCN → learned target and GCN → original compare? |

## 1. Evaluation protocol and configuration

`TEST_CONFIGURATIONS` selects the current-window and dual-window modalities. Each selected target must originally be observed so that its pre-masking embedding can serve as ground truth. The target is then artificially masked. The available context is used for GCN imputation and, where enabled, the learned cross-modal target. Each model is evaluated against **its own** original embedding. Use `BATCH_LIMIT=None` for full-split reporting.

**Pairing qualification:** The paired CL-effect calculation joins records by `sample_id`. If IDs are based on evaluation order rather than unique encounter identifiers, the two datasets must have independently verified identical input order and eligibility selection. Matching identifier strings alone does not establish genuine clinical pairing. Different windows may also involve different cohorts, available context or model configurations.

In [ ]:
from pathlib import Path
import os
import sys
import math
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path('/data/mm-healthfusion-m3care-hospitalisation')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EVAL_SPLIT = 'test'
BATCH_LIMIT = None  # Set to a small integer for a quick test; None = full split.
OUT_DIR = PROJECT_ROOT / 'outputs' / 'm3care_imputation_quality'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Visual style follows the supplied presentation code.
RUN_COLORS = {
    'Model A': '#6D2E8E',
    'Model B': '#0E7C84',
}
CHAIN_COLORS = {
    'Learned target -> true': '#6D2E8E',
    'GCN -> learned target': '#0E7C84',
    'GCN -> true': '#BE7C1C',
}

plt.rcParams.update({
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
})

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DEVICE:', DEVICE)
print('OUT_DIR:', OUT_DIR)

EVAL_SPLIT = 'test'

# Four paths per window: Model A checkpoint/config and Model B checkpoint/config.
curra_ckpt = "/data/logs/ablation/m3care/hospitalisation/currWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_normloss_w0_01_imputedMiss_maskFusion_EXT_ST_currWin_3_1hosp/checkpoint_best.ckpt"
curra_cfg = "/data/logs/ablation/m3care/hospitalisation/currWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_normloss_w0_01_imputedMiss_maskFusion_EXT_ST_currWin_3_1hosp/config.yaml"
currb_ckpt = "/data/logs/ablation/m3care/hospitalisation/currWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_hospBest_CL_finetune_missW0_5_EXT_ST_currWin_3_1hosp/checkpoint_best.ckpt"
currb_cfg = "/data/logs/ablation/m3care/hospitalisation/currWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_hospBest_CL_finetune_missW0_5_EXT_ST_currWin_3_1hosp/config.yaml"
duala_ckpt = "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_imputedMiss_maskFusion_3_1hosp/checkpoint_best.ckpt"
duala_cfg = "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_imputedMiss_maskFusion_3_1hosp/config.yaml"
dualb_ckpt = "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_hospBest_CL_1S_missW0_5_EXT_ST_dualWin_3_1hosp/checkpoint_best.ckpt"
dualb_cfg = "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_hospBest_CL_1S_missW0_5_EXT_ST_dualWin_3_1hosp/config.yaml"


MODEL_PATHS = {
    'currWindow': {
        'model_A': {'checkpoint': Path(curra_ckpt), 'config': Path(curra_cfg)},
        'model_B': {'checkpoint': Path(currb_ckpt), 'config': Path(currb_cfg)},
    },
    'dualWindow': {
        'model_A': {'checkpoint': Path(duala_ckpt), 'config': Path(duala_cfg)},
        'model_B': {'checkpoint': Path(dualb_ckpt), 'config': Path(dualb_cfg)},
    },
}

# Select the hospitalisation combinations to evaluate.
TEST_CONFIGURATIONS = [
    ('currWindow', ['cxr_curr', 'ecg_curr']),
    ('dualWindow', ['cxr_curr', 'ecg_curr', 'notes_prev']),
]

REQUIRE_ALL_CONTEXT_MODALITIES = False

print('TEST_CONFIGURATIONS:', TEST_CONFIGURATIONS)

In [ ]:
import yaml
from src.data.splits import get_dataloaders
from src.training.lightning import MMModel


def load_cfg(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Config does not exist: {path}')
    if path.suffix.lower() not in {'.yaml', '.yml'}:
        raise ValueError(f'Expected YAML config, got: {path}')
    with path.open('r', encoding='utf-8') as f:
        return yaml.safe_load(f) or {}


def infer_feature_dims(batch, model_cfg):
    modality_names = list(model_cfg.get('modality_names', [])) or ['tabular', 'ecg', 'cxr']
    aliases = {
        'tabular': ['tab_feats', 'static_feats'],
        'static': ['static_feats', 'tab_feats'],
    }
    dims = {}
    for modality in modality_names:
        found = None
        for key in aliases.get(modality, [f'{modality}_feats']):
            if key in batch and torch.is_tensor(batch[key]):
                found = int(batch[key].shape[-1])
                break
        if found is None:
            raise KeyError(
                f'Cannot infer feature dimension for {modality!r}. '
                f'Available batch keys: {sorted(batch.keys())}'
            )
        dims[modality] = found
    if 'tabular' in dims:
        dims.setdefault('static', dims['tabular'])
    if 'static' in dims:
        dims.setdefault('tabular', dims['static'])
    return dims


def build_eval_loader(cfg, split):
    loaders = get_dataloaders(cfg)
    if split not in loaders:
        raise KeyError(f'Split {split!r} not available: {list(loaders)}')
    loader = loaders[split]
    return DataLoader(
        loader.dataset,
        batch_size=loader.batch_size,
        shuffle=False,
        num_workers=loader.num_workers,
        collate_fn=loader.collate_fn,
        pin_memory=loader.pin_memory,
        drop_last=False,
    )


def load_model(checkpoint_path, config_path, split):
    cfg = load_cfg(config_path)
    model_cfg = dict(cfg.get('model', {}))
    model_cfg.pop('load_from', None)
    fusion_method = str(model_cfg.get('fusion_method', model_cfg.get('name', 'm3care'))).lower()
    if fusion_method != 'm3care':
        raise ValueError(f'Expected M3Care, got {fusion_method!r}')

    loader = build_eval_loader(cfg, split)
    first_batch = next(iter(loader))
    feature_dims = infer_feature_dims(first_batch, model_cfg)

    lit_model = MMModel(
        model=None,
        learning_rate=float(cfg.get('train', {}).get('learning_rate', 3e-4)),
        weight_decay=float(cfg.get('train', {}).get('weight_decay', 1e-4)),
        pos_weight=None,
        scheduler_patience=None,
        scheduler_monitor='val_mcc',
        fusion_method=fusion_method,
        fusion_config=model_cfg,
        feature_dims=feature_dims,
        modality_dropout_config={},
    )
    lit_model.load_pretrained_weights(str(checkpoint_path), strict=True)
    lit_model.to(DEVICE)
    lit_model.eval()
    return lit_model, loader, cfg

In [ ]:
def move_batch_to_device(batch):
    return {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}


def model_masks(model, batch, encoded):
    return model._modality_mask(batch, encoded)


def prefix_for_modality(modality):
    return 'tab' if modality == 'tabular' else modality


def mask_batch_modality(batch, modality, row_mask):
    edited = dict(batch)
    prefix = prefix_for_modality(modality)
    for suffix in ('missing', 'lengths', 'feats', 'time'):
        key = f'{prefix}_{suffix}'
        if key in edited and torch.is_tensor(edited[key]):
            edited[key] = edited[key].clone()
    for suffix, value in [('missing', 1), ('lengths', 0), ('feats', 0), ('time', 0)]:
        key = f'{prefix}_{suffix}'
        if key in edited:
            edited[key][row_mask] = value
    return edited


def context_modalities_for_model(model, target_modality):
    names = list(getattr(model.imputation, 'modality_names', []))
    return [m for m in names if m != target_modality]


def eligible_rows(masks, target_modality, model, require_all_context=False):
    if target_modality not in masks:
        raise KeyError(f'Target modality {target_modality!r} not found in masks: {list(masks)}')
    target_present = masks[target_modality].reshape(-1).bool()
    context_modalities = context_modalities_for_model(model, target_modality)
    context_available = [masks[m].reshape(-1).bool() for m in context_modalities if m in masks]
    if not context_available:
        return target_present, context_modalities
    stacked = torch.stack(context_available, dim=0)
    context_ok = stacked.all(dim=0) if require_all_context else stacked.any(dim=0)
    return target_present & context_ok, context_modalities


def _scalarize_id(value, i):
    if torch.is_tensor(value):
        v = value[i]
        if torch.is_tensor(v):
            v = v.detach().cpu().item() if v.numel() == 1 else tuple(v.detach().cpu().reshape(-1).tolist())
        return str(v)
    if isinstance(value, (list, tuple)):
        return str(value[i])
    return str(value)


def sample_ids_from_batch(batch, start_index, batch_size):
    """Use dataset identifiers when available; otherwise fall back to stable evaluation order."""
    preferred = [
        'sample_id', 'patient_id', 'subject_id', 'stay_id', 'hadm_id',
        'icustay_id', 'study_id', 'row_id', 'index',
    ]
    for key in preferred:
        if key in batch:
            value = batch[key]
            try:
                ids = [_scalarize_id(value, i) for i in range(batch_size)]
                if len(set(ids)) == len(ids):
                    return ids
            except Exception:
                pass
    return [f'eval_order_{start_index + i:08d}' for i in range(batch_size)]


def _gcn_impute_selected_target(model, masked_encoded, masked_masks, target_modality):
    imp = model.imputation
    if target_modality not in imp.modalities_to_impute:
        raise ValueError(
            f'{target_modality!r} is not in modalities_to_impute: {imp.modalities_to_impute}'
        )
    sim_list, mask_list = [], []
    for idx, name in enumerate(imp.modality_names):
        x = masked_encoded[name]
        mask = masked_masks[name]
        mask2 = mask * mask.transpose(0, 1)
        sim, _ = imp._build_similarity(x, imp.simiProj[idx], imp.eps_params[idx])
        sim = sim * mask2
        sim_list.append(sim)
        mask_list.append(mask2)
    sum_masks = torch.stack(mask_list, dim=0).sum(dim=0).clamp_min(1.0)
    similar_score = torch.stack(sim_list, dim=0).sum(dim=0) / sum_masks
    threshold = imp.sigmoid(imp.threshold).reshape(-1)[0]
    similar_score = imp.relu(similar_score - threshold)
    bin_mask = similar_score > 0
    similar_score = similar_score + bin_mask * threshold.detach()
    target_idx = imp.modalities_to_impute.index(target_modality)
    return imp._refine_with_gcn(
        similar_score,
        masked_encoded[target_modality],
        masked_masks[target_modality],
        imp.GCN[target_idx],
        imp.GCN_2[target_idx],
        imp.weight1[target_idx],
        imp.weight2[target_idx],
    )[:2]


def raw_metric_vectors(pred, true):
    pred, true = pred.float(), true.float()
    cosine = F.cosine_similarity(pred, true, dim=-1, eps=1e-8)
    mse = ((pred - true) ** 2).mean(dim=-1)
    mae = (pred - true).abs().mean(dim=-1)
    true_norm = true.norm(dim=-1)
    pred_norm = pred.norm(dim=-1)
    rel_norm = (pred_norm - true_norm).abs() / true_norm.clamp_min(1e-8)
    return cosine, mse, mae, rel_norm


def projected_pair(anchor, target, imp):
    anchor_z = F.normalize(imp.contrastive_projector(anchor), p=2, dim=-1, eps=1e-8)
    target_z = F.normalize(imp.contrastive_target_projector(target.detach()), p=2, dim=-1, eps=1e-8)
    return anchor_z, target_z


def projected_diagnostics(anchor_z, target_z, temperature):
    n = anchor_z.size(0)
    sim = anchor_z @ target_z.T
    labels = torch.arange(n, device=sim.device)
    logits = sim / temperature
    row_ce = F.cross_entropy(logits, labels, reduction='none')
    col_ce = F.cross_entropy(logits.T, labels, reduction='none')
    nce = 0.5 * (row_ce + col_ce)
    retrieval = (sim.argmax(dim=1) == labels).float()
    if n > 1:
        neg_mask = ~torch.eye(n, dtype=torch.bool, device=sim.device)
        neg = sim[neg_mask].reshape(n, n - 1)
        gap = sim.diag() - neg.mean(dim=1)
    else:
        gap = torch.full((n,), float('nan'), device=sim.device)
    return sim.diag(), retrieval, gap, nce


def learned_target_available(model):
    imp = model.imputation
    return (
        bool(getattr(imp, 'learned_cross_modal_target', False))
        and hasattr(imp, 'learned_target_networks')
        and hasattr(imp, 'contrastive_target_projector')
        and hasattr(imp, 'contrastive_projector')
    )


def learned_target_for_batch(model, encoded, masks, target_modality):
    if not learned_target_available(model):
        return None
    target, _ = model.imputation._build_cross_modal_target(encoded, masks, target_modality)
    return target.detach()


def evaluate_one_batch(lit_model, batch, target_modality, start_index, require_all_context=False):
    model = lit_model.model
    with torch.no_grad():
        original_batch = move_batch_to_device(batch)
        original_encoded = model.encode(original_batch)
        original_masks = model_masks(model, original_batch, original_encoded)
        eligible, context_modalities = eligible_rows(original_masks, target_modality, model, require_all_context)
        if not bool(eligible.any().item()):
            return None, 0

        n_batch = int(eligible.sum().item())
        true_target = original_encoded[target_modality][eligible].detach()
        sample_ids_all = sample_ids_from_batch(original_batch, start_index, len(eligible))
        sample_ids = [sample_ids_all[i] for i, ok in enumerate(eligible.detach().cpu().tolist()) if ok]

        masked_batch = mask_batch_modality(original_batch, target_modality, eligible)
        masked_encoded = model.encode(masked_batch)
        masked_masks = model_masks(model, masked_batch, masked_encoded)
        x0_all, x_final_all = _gcn_impute_selected_target(model, masked_encoded, masked_masks, target_modality)
        x0 = x0_all[eligible].detach()
        x_final = x_final_all[eligible].detach()
        teacher_all = learned_target_for_batch(model, masked_encoded, masked_masks, target_modality)
        teacher = None if teacher_all is None else teacher_all[eligible].detach()

    return {
        'sample_id': sample_ids,
        'true_target': true_target,
        'x0': x0,
        'x_final': x_final,
        'teacher': teacher,
    }, n_batch


def evaluate_model(lit_model, loader, model_name, target_modality, require_all_context=False, batch_limit=None):
    rows = []
    processed_batches = 0
    total_eligible = 0
    imp = lit_model.model.imputation
    teacher_enabled = learned_target_available(lit_model.model)
    start_index = 0

    for batch in loader:
        if batch_limit is not None and processed_batches >= batch_limit:
            break
        batch_size = next(v.shape[0] for v in batch.values() if torch.is_tensor(v) and v.ndim > 0)
        result, n_batch = evaluate_one_batch(
            lit_model, batch, target_modality, start_index,
            require_all_context=require_all_context,
        )
        start_index += batch_size
        processed_batches += 1
        if result is None:
            continue
        total_eligible += n_batch

        true_target = result['true_target']
        x0 = result['x0']
        x_final = result['x_final']
        ids = result['sample_id']

        for rep_name, pred in [('gcn_x0', x0), ('final_imputed', x_final)]:
            cosine, mse, mae, rel_norm = raw_metric_vectors(pred, true_target)
            for i, sid in enumerate(ids):
                rows.append({
                    'sample_id': sid,
                    'model': model_name,
                    'representation': rep_name,
                    'gcn_vs_true_raw_cosine': float(cosine[i].cpu()),
                    'gcn_vs_true_raw_mse': float(mse[i].cpu()),
                    'gcn_vs_true_raw_mae': float(mae[i].cpu()),
                    'gcn_vs_true_raw_relative_norm_error': float(rel_norm[i].cpu()),
                    'teacher_available': teacher_enabled,
                })

        teacher = result['teacher']
        if teacher is None or x0.shape[0] < 2:
            continue

        teacher_cos, teacher_mse, teacher_mae, teacher_rel = raw_metric_vectors(teacher, true_target)
        gct_cos, gct_mse, gct_mae, gct_rel = raw_metric_vectors(x0, teacher)

        teacher_z, true_z = projected_pair(teacher, true_target, imp)
        gct_z, teacher_t_z = projected_pair(x0, teacher, imp)
        gct_true_z, true_target_z = projected_pair(x0, true_target, imp)

        t_proj_cos, t_retr, t_gap, t_nce = projected_diagnostics(
            teacher_z, true_z,
            float(getattr(imp, 'learned_target_contrastive_temperature', imp.contrastive_temperature)),
        )
        gt_proj_cos, gt_retr, gt_gap, gt_nce = projected_diagnostics(
            gct_z, teacher_t_z, float(imp.contrastive_temperature)
        )
        gtrue_proj_cos, gtrue_retr, gtrue_gap, _ = projected_diagnostics(
            gct_true_z, true_target_z, float(imp.contrastive_temperature)
        )

        for i, sid in enumerate(ids):
            rows.append({
                'sample_id': sid,
                'model': model_name,
                'representation': 'teacher_chain',
                'teacher_vs_true_raw_cosine': float(teacher_cos[i].cpu()),
                'teacher_vs_true_raw_mse': float(teacher_mse[i].cpu()),
                'teacher_vs_true_raw_mae': float(teacher_mae[i].cpu()),
                'teacher_vs_true_raw_relative_norm_error': float(teacher_rel[i].cpu()),
                'teacher_vs_true_projected_cosine': float(t_proj_cos[i].cpu()),
                'teacher_vs_true_retrieval_accuracy': float(t_retr[i].cpu()),
                'teacher_vs_true_pos_neg_gap': float(t_gap[i].cpu()),
                'teacher_target_infonce': float(t_nce[i].cpu()),
                'gcn_vs_teacher_raw_cosine': float(gct_cos[i].cpu()),
                'gcn_vs_teacher_raw_mse': float(gct_mse[i].cpu()),
                'gcn_vs_teacher_raw_mae': float(gct_mae[i].cpu()),
                'gcn_vs_teacher_raw_relative_norm_error': float(gct_rel[i].cpu()),
                'gcn_vs_teacher_projected_cosine': float(gt_proj_cos[i].cpu()),
                'gcn_vs_teacher_retrieval_accuracy': float(gt_retr[i].cpu()),
                'gcn_vs_teacher_pos_neg_gap': float(gt_gap[i].cpu()),
                'gcn_to_teacher_infonce': float(gt_nce[i].cpu()),
                'gcn_vs_true_projected_cosine': float(gtrue_proj_cos[i].cpu()),
                'gcn_vs_true_retrieval_accuracy': float(gtrue_retr[i].cpu()),
                'gcn_vs_true_pos_neg_gap': float(gtrue_gap[i].cpu()),
            })

    return pd.DataFrame(rows), processed_batches, total_eligible


def run_one_window(window, target_modalities, model_paths, split='val', require_all_context=False, batch_limit=None):
    paths = model_paths[window]
    model_a, loader_a, _ = load_model(paths['model_A']['checkpoint'], paths['model_A']['config'], split)
    model_b, loader_b, _ = load_model(paths['model_B']['checkpoint'], paths['model_B']['config'], split)
    chunks = []
    for target_modality in target_modalities:
        for label, model, loader in [('Model A', model_a, loader_a), ('Model B', model_b, loader_b)]:
            if target_modality not in model.model.imputation.modalities_to_impute:
                raise ValueError(
                    f'{target_modality!r} is not imputable by {label} in {window}: '
                    f'{model.model.imputation.modalities_to_impute}'
                )
        ra, ba, na = evaluate_model(model_a, loader_a, 'Model A', target_modality, require_all_context, batch_limit)
        rb, bb, nb = evaluate_model(model_b, loader_b, 'Model B', target_modality, require_all_context, batch_limit)
        r = pd.concat([ra, rb], ignore_index=True)
        r.insert(0, 'window', window)
        r.insert(1, 'target_modality', target_modality)
        chunks.append(r)
        print(f'{window} / {target_modality}: Model A n={na}, Model B n={nb}')
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

## 2. Run the selected window/modality combinations

Each selected window is loaded once and its selected target modalities are evaluated using the original observed embedding as ground truth. The GCN and learned-target network (when present) receive the same masked context.

In [ ]:
all_chunks = []

for window, target_modalities in TEST_CONFIGURATIONS:
    if window not in MODEL_PATHS:
        raise KeyError(f'Unknown window {window!r}; choose from {list(MODEL_PATHS)}')

    paths = MODEL_PATHS[window]
    model_a, loader_a, cfg_a = load_model(paths['model_A']['checkpoint'], paths['model_A']['config'], EVAL_SPLIT)
    model_b, loader_b, cfg_b = load_model(paths['model_B']['checkpoint'], paths['model_B']['config'], EVAL_SPLIT)

    for target_modality in target_modalities:
        for label, model in [('Model A', model_a), ('Model B', model_b)]:
            if target_modality not in model.model.imputation.modalities_to_impute:
                raise ValueError(
                    f'{target_modality!r} is not imputable by {label} in {window}: '
                    f'{model.model.imputation.modalities_to_impute}'
                )

        ra, ba, na = evaluate_model(
            model_a, loader_a, 'Model A', target_modality,
            REQUIRE_ALL_CONTEXT_MODALITIES, BATCH_LIMIT
        )
        rb, bb, nb = evaluate_model(
            model_b, loader_b, 'Model B', target_modality,
            REQUIRE_ALL_CONTEXT_MODALITIES, BATCH_LIMIT
        )

        r = pd.concat([ra, rb], ignore_index=True)
        r.insert(0, 'window', window)
        r.insert(1, 'target_modality', target_modality)
        all_chunks.append(r)
        print(f'{window} / {target_modality}: Model A n={na}; Model B n={nb}')

results = pd.concat(all_chunks, ignore_index=True) if all_chunks else pd.DataFrame()
print("Total evaluation rows:", len(results))

# results.head() # Commented out to avoid data leakage

## 3. Summary tables

The evaluation records one row per eligible observation per representation. Summary means and approximate 95% confidence intervals are calculated from observation-level values, not from unweighted averages of batch means. The primary model means are calculated separately. The paired CL-effect interval is calculated from **within-observation differences**. The two interval types answer different questions.

In [ ]:
def mean_ci(values):
    values = pd.Series(values).dropna().astype(float)
    if len(values) == 0:
        return np.nan, np.nan, np.nan, 0
    mean = float(values.mean())
    if len(values) > 1:
        se = float(values.std(ddof=1)) / math.sqrt(len(values))
        lo, hi = mean - 1.96 * se, mean + 1.96 * se
    else:
        lo = hi = np.nan
    return mean, lo, hi, len(values)


def paired_ci(values):
    return mean_ci(values)


def primary_summary(results):
    rows = []
    for keys, grp in results.groupby([c for c in ['window', 'target_modality', 'model'] if c in results.columns]):
        if not isinstance(keys, tuple):
            keys = (keys,)
        key_dict = dict(zip([c for c in ['window', 'target_modality', 'model'] if c in results.columns], keys))
        gcn = grp[grp['representation'] == 'gcn_x0']
        m, lo, hi, n = mean_ci(gcn['gcn_vs_true_raw_cosine'])
        mse_m, _, _, _ = mean_ci(gcn['gcn_vs_true_raw_mse'])
        mae_m, _, _, _ = mean_ci(gcn['gcn_vs_true_raw_mae'])
        rn_m, _, _, _ = mean_ci(gcn['gcn_vs_true_raw_relative_norm_error'])
        rows.append({**key_dict, 'n': n, 'cosine_mean': m, 'cosine_ci_low': lo, 'cosine_ci_high': hi,
                     'mse_mean': mse_m, 'mae_mean': mae_m, 'relative_norm_error_mean': rn_m})
    return pd.DataFrame(rows)


def teacher_summary(results):
    rows = []
    if 'teacher_chain' not in set(results.get('representation', [])):
        return pd.DataFrame()
    for keys, grp in results[results['representation'] == 'teacher_chain'].groupby([c for c in ['window', 'target_modality', 'model'] if c in results.columns]):
        if not isinstance(keys, tuple): keys = (keys,)
        key_dict = dict(zip([c for c in ['window','target_modality','model'] if c in results.columns], keys))
        row = dict(key_dict)
        for prefix, cols in [
            ('teacher_vs_true', ['teacher_vs_true_raw_cosine','teacher_vs_true_raw_mse','teacher_vs_true_raw_mae','teacher_vs_true_projected_cosine','teacher_vs_true_retrieval_accuracy','teacher_target_infonce']),
            ('gcn_vs_teacher', ['gcn_vs_teacher_raw_cosine','gcn_vs_teacher_raw_mse','gcn_vs_teacher_raw_mae','gcn_vs_teacher_projected_cosine','gcn_vs_teacher_retrieval_accuracy','gcn_to_teacher_infonce']),
            ('gcn_vs_true_projected', ['gcn_vs_true_projected_cosine','gcn_vs_true_retrieval_accuracy']),
        ]:
            for c in cols:
                m, lo, hi, n = mean_ci(grp[c])
                row[f'{c}_mean'] = m; row[f'{c}_ci_low'] = lo; row[f'{c}_ci_high'] = hi
        row['n'] = int(len(grp))
        rows.append(row)
    return pd.DataFrame(rows)


def paired_effect_summary(results):
    rows = []
    base_cols = ['window','target_modality'] if 'window' in results.columns else ['target_modality']
    for keys, grp in results.groupby(base_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        key_dict = dict(zip(base_cols, keys))
        a = grp[(grp['model']=='Model A') & (grp['representation']=='gcn_x0')][['sample_id','gcn_vs_true_raw_cosine']].rename(columns={'gcn_vs_true_raw_cosine':'cos_a'})
        b = grp[(grp['model']=='Model B') & (grp['representation']=='gcn_x0')][['sample_id','gcn_vs_true_raw_cosine']].rename(columns={'gcn_vs_true_raw_cosine':'cos_b'})
        merged = a.merge(b, on='sample_id', how='inner', validate='one_to_one')
        delta = merged['cos_b'] - merged['cos_a']
        mean, lo, hi, n = paired_ci(delta)
        rows.append({**key_dict, 'n_paired': n, 'delta_cosine_mean': mean, 'delta_cosine_ci_low': lo, 'delta_cosine_ci_high': hi})
    return pd.DataFrame(rows)

In [ ]:
primary = primary_summary(results)
effect = paired_effect_summary(results)
teacher = teacher_summary(results)

print('=== Primary imputation quality ===')
display(primary)

print('=== Paired CL effect (Model B - Model A) ===')
display(effect)

print('=== Learned-target teacher chain ===')
display(teacher)

### How to read the accompanying tables

| Field | Interpretation |
|---|---|
| `n` / `n_paired` | Number of eligible observations for a model / number of matched pairs for the CL-effect analysis. |
| `cosine_mean`, `cosine_ci_low`, `cosine_ci_high` | Mean raw GCN → original cosine similarity and approximate 95% CI for each model; **higher** means stronger directional agreement. |
| `mse_mean`, `mae_mean` | Mean coordinate-wise squared / absolute embedding error; **lower** is better, but values depend on embedding scale. |
| `relative_norm_error_mean` | Relative difference in embedding magnitudes; **lower** is better. |
| `delta_cosine_mean`, `delta_cosine_ci_low`, `delta_cosine_ci_high` | Mean **With CL − Without CL** cosine difference and approximate 95% CI computed from matched differences; positive favours CL. |
| `teacher_vs_true_*` | **Learned target → original** metrics (the code retains `teacher` in internal field names). |
| `gcn_vs_teacher_*` | **GCN → learned target** metrics (internal field names retain `teacher`). |
| `gcn_vs_true_projected_*` | Similarity after contrastive projection; **not** the raw GCN → original reconstruction metric. |

The learned-target diagnostics are available only where the corresponding network is enabled. A CI crossing zero means the estimated paired effect is not clearly separated from zero under the reported interval method; it does not prove the absence of an effect.

## 4. Evaluation figures

All figures retain the notebook's existing palette, confidence-interval conventions, transparent backgrounds and export settings. In model-comparison plots, purple means **Without CL** and teal means **With CL**. In Figure 3, the colours represent **relationships**, not separate model runs.

**Figure 1 — Absolute GCN imputation quality.** For each selected window/modality, the bars show separate model means of raw GCN → original cosine similarity with approximate 95% CIs. Higher values indicate greater directional agreement. These intervals are not paired-difference intervals.

**Figure 2 — Paired effect of CL.** Each bar shows the mean matched-observation difference **With CL − Without CL** in raw GCN → original cosine similarity, with an approximate 95% CI of the paired differences. Positive favours CL; negative favours Without CL. Pairing is justified only after observation alignment is verified.

**Figure 3 — Learned-target alignment diagnostic.** Purple = **Learned target → original** (quality of the learned target), teal = **GCN → learned target** (alignment with that target), and orange = **GCN → original** (actual imputation alignment). All three are **raw embedding-space cosine similarities**. The orange bar should agree with the With-CL result for the same configuration in Figure 1. A strong learned target does not by itself demonstrate successful transfer to the GCN.

**Window comparisons.** Dashed vertical separators distinguish current-window and dual-window groups. Differences may reflect temporal context, eligible cohort and model configuration as well as CL. Avoid interpreting between-window differences as the isolated effect of CL.

In [ ]:
def _savefig(fig, filename):
    path = OUT_DIR / filename
    fig.savefig(
        path,
        dpi=600,
        transparent=True,
        bbox_inches="tight",
    )
    print("Saved:", path)


def _bar_value_label(ax, x, value, upper_error=0.0):
    if pd.notna(value):
        ax.text(
            x,
            value + upper_error + 0.012,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=6,
            color="#404040",
        )


def _add_window_separator_and_labels(
    ax,
    keys,
    *,
    y_text=0.93,
    separator_color="#808080",
):
    """
    Add a vertical separator between currWindow and dualWindow and,
    when both are present, add labels above the two regions.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axis to modify.

    keys : list of tuples
        Plot ordering, e.g.
        [
            ("currWindow", "cxr_curr"),
            ("currWindow", "ecg_curr"),
            ("dualWindow", "cxr_curr"),
            ("dualWindow", "ecg_curr"),
        ]

    y_text : float
        Vertical position of the window labels in axis coordinates.
    """

    if not keys:
        return

    # If this is a single-window plot, there is nothing to separate.
    if not isinstance(keys[0], tuple) or len(keys[0]) < 2:
        return

    windows = [key[0] for key in keys]

    n_curr = sum(
        window == "currWindow"
        for window in windows
    )

    n_dual = sum(
        window == "dualWindow"
        for window in windows
    )

    # Only add the separator when both windows are present.
    if n_curr == 0 or n_dual == 0:
        return

    # Separator goes between the final currWindow group and
    # the first dualWindow group.
    ax.axvline(
        x=n_curr - 0.5,
        linestyle="--",
        linewidth=1,
        color=separator_color,
        alpha=0.7,
        zorder=0,
    )

    # Centre positions of the two window regions.
    curr_center = (n_curr - 1) / 2
    dual_center = n_curr + (n_dual - 1) / 2

    ax.text(
        curr_center,
        y_text,
        "currWindow",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="bottom",
        fontsize=8,
        color="#404040",
    )

    ax.text(
        dual_center,
        y_text,
        "dualWindow",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="bottom",
        fontsize=8,
        color="#404040",
    )


def _get_plot_keys(summary):
    """
    Return plotting keys in the same order used on the x-axis.
    """
    if summary.empty:
        return []

    if "window" in summary.columns:
        return list(
            summary[
                ["window", "target_modality"]
            ]
            .drop_duplicates()
            .itertuples(index=False, name=None)
        )

    return [
        (target_modality,)
        for target_modality
        in summary["target_modality"].drop_duplicates()
    ]


def plot_primary_imputation(
    results,
    title="GCN imputation quality across modalities",
    filename="plot1_gcn_vs_true.png",
):
    sm = primary_summary(results)

    if sm.empty:
        print("No primary results to plot.")
        return

    keys = _get_plot_keys(sm)

    if "window" in sm.columns:
        labels = [
            # f"{window}\n{target_modality}"
            f"{target_modality}"
            for window, target_modality in keys
        ]
    else:
        labels = [
            target_modality
            for (target_modality,) in keys
        ]

    x = np.arange(len(keys))

    model_labels = {
        "Model A": "Without CL",
        "Model B": "With CL",
    }

    runs = ["Model A", "Model B"]

    model_handles = [
        Patch(
            facecolor=RUN_COLORS[m],
            edgecolor="white",
            label=model_labels[m],
        )
        for m in runs
    ]

    width = min(
        0.36,
        0.72 / len(runs),
    )

    fig, ax = plt.subplots(
        figsize=(4.5, 3.4),
        dpi=600,
    )

    fig.patch.set_alpha(0.0)
    ax.patch.set_alpha(0.0)

    for ri, run in enumerate(runs):

        for i, key in enumerate(keys):

            q = sm[sm["model"] == run]

            if "window" in sm.columns:
                q = q[
                    (q["window"] == key[0])
                    & (
                        q["target_modality"]
                        == key[1]
                    )
                ]
            else:
                q = q[
                    q["target_modality"]
                    == key[0]
                ]

            if q.empty:
                continue

            r = q.iloc[0]

            p = float(r["cosine_mean"])

            lo = (
                p
                - float(r["cosine_ci_low"])
            )

            hi = (
                float(r["cosine_ci_high"])
                - p
            )

            offset = (
                ri
                - (len(runs) - 1) / 2
            ) * width

            ax.bar(
                x[i] + offset,
                p,
                width * 0.92,
                color=RUN_COLORS[run],
                edgecolor="white",
                linewidth=0.5,
                yerr=[[lo], [hi]],
                capsize=2,
                error_kw=dict(
                    ecolor="lightgray",
                    elinewidth=1,
                ),
                alpha=0.9,
            )

            _bar_value_label(
                ax,
                x[i] + offset,
                p,
                hi,
            )

    # ------------------------------------------------------------
    # Separate currWindow and dualWindow
    # ------------------------------------------------------------
    if "window" in sm.columns:
        _add_window_separator_and_labels(
            ax,
            keys,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        labels,
        fontsize=8,
    )

    ax.tick_params(
        axis="x",
        length=0,
    )

    ax.set_ylabel(
        "Raw cosine similarity"
    )

    ax.set_title(title)

    ax.set_ylim(
        min(
            0,
            float(sm["cosine_ci_low"].min())
            - 0.05,
        ),
        min(
            1.08,
            float(sm["cosine_ci_high"].max())
            + 0.10,
        ),
    )

    ax.legend(
        handles=model_handles,
        # title="Model",
        title_fontsize=8,
        fontsize=8,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.20),
        frameon=False,
        ncol=2,
        handletextpad=0.6,
        borderaxespad=0.0,
    )

    plt.tight_layout()

    _savefig(
        fig,
        filename,
    )

    plt.show()


def plot_paired_cl_effect(
    results,
    title="Change in GCN imputation quality with CL",
    filename="plot2_cl_effect.png",
):
    sm = paired_effect_summary(results)

    if sm.empty:
        print(
            "No paired baseline/with-CL results to plot."
        )
        return

    # ------------------------------------------------------------
    # Determine plotting order
    # ------------------------------------------------------------
    if "window" in sm.columns:
        keys = list(
            sm[
                ["window", "target_modality"]
            ]
            .drop_duplicates()
            .itertuples(
                index=False,
                name=None,
            )
        )

        labels = [
            # f"{window}\n{target_modality}"
            f"{target_modality}"
            for window, target_modality in keys
        ]

    else:
        keys = [
            (target_modality,)
            for target_modality
            in sm["target_modality"].drop_duplicates()
        ]

        labels = [
            target_modality
            for (target_modality,) in keys
        ]

    # Reorder summary according to keys.
    ordered_rows = []

    for key in keys:

        if "window" in sm.columns:
            q = sm[
                (sm["window"] == key[0])
                & (
                    sm["target_modality"]
                    == key[1]
                )
            ]
        else:
            q = sm[
                sm["target_modality"]
                == key[0]
            ]

        if not q.empty:
            ordered_rows.append(
                q.iloc[0]
            )

    if not ordered_rows:
        print(
            "No paired summary rows available."
        )
        return

    sm = pd.DataFrame(
        ordered_rows
    ).reset_index(drop=True)

    x = np.arange(len(sm))

    fig, ax = plt.subplots(
        figsize=(5, 3.4),
        dpi=600,
    )

    fig.patch.set_alpha(0.0)
    ax.patch.set_alpha(0.0)

    for i, r in sm.iterrows():

        p = float(
            r["delta_cosine_mean"]
        )

        lo = (
            p
            - float(
                r["delta_cosine_ci_low"]
            )
        )

        hi = (
            float(
                r["delta_cosine_ci_high"]
            )
            - p
        )

        ax.bar(
            x[i],
            p,
            0.60,
            color=RUN_COLORS["Model B"],
            edgecolor="white",
            linewidth=0.5,
            yerr=[[lo], [hi]],
            capsize=2,
            error_kw=dict(
                ecolor="lightgray",
                elinewidth=1,
            ),
            alpha=0.9,
        )

        if p >= 0:

            _bar_value_label(
                ax,
                x[i],
                p,
                hi,
            )

        else:

            ax.text(
                x[i],
                p - lo - 0.012,
                f"{p:.3f}",
                ha="center",
                va="top",
                fontsize=6,
                color="#404040",
            )

    # Zero reference line.
    ax.axhline(
        0,
        color="#404040",
        linewidth=0.8,
    )

    # ------------------------------------------------------------
    # Separate currWindow and dualWindow
    # ------------------------------------------------------------
    if "window" in sm.columns:
        _add_window_separator_and_labels(
            ax,
            keys,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        labels,
        fontsize=8,
    )

    ax.tick_params(
        axis="x",
        length=0,
    )

    ax.set_ylabel(
        "CL - no CL: Δ cosine similarity"
    )

    ax.set_title(title)

    ymax = (
        max(
            abs(
                float(
                    sm[
                        "delta_cosine_ci_low"
                    ].min()
                )
            ),
            abs(
                float(
                    sm[
                        "delta_cosine_ci_high"
                    ].max()
                )
            ),
        )
        if len(sm)
        else 0.1
    )

    ax.set_ylim(
        -max(0.05, ymax * 1.35),
        max(0.05, ymax * 1.35),
    )

    plt.tight_layout()

    _savefig(
        fig,
        filename,
    )

    plt.show()


In [ ]:

def _add_window_separator_and_labels_chain(
    ax,
    keys,
    x_positions,
    y_text=0.93,
):
    """
    Add a separator and centred window labels using
    the actual x-axis positions of modality groups.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axis containing the bar chart.
    keys : list of tuples
        Each tuple contains (window, target_modality).
    x_positions : array-like
        Actual x-axis positions of modality groups.
    y_text : float
        Vertical position of window labels in axes coordinates.
    """

    windows = [str(k[0]) for k in keys]
    x_positions = np.asarray(x_positions, dtype=float)

    if len(windows) != len(x_positions):
        raise ValueError(
            "keys and x_positions must have the same length."
        )

    # Identify consecutive groups belonging to each window
    groups = []
    start = 0

    for i in range(1, len(windows)):
        if windows[i] != windows[i - 1]:
            groups.append((windows[start], start, i - 1))
            start = i

    groups.append((windows[start], start, len(windows) - 1))

    # Window labels
    for window, first, last in groups:

        centre_x = (
            x_positions[first] + x_positions[last]
        ) / 2

        label = {
            "currWindow": "currWindow",
            "dualWindow": "dualWindow",
        }.get(window, window)

        ax.text(
            centre_x,
            y_text,
            label,
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="center",
            fontsize=9,
            color="black",
            fontweight="medium",
            clip_on=False,
        )

    # Separators between consecutive windows
    for (_, _, last), (_, first, _) in zip(
        groups[:-1], groups[1:]
    ):

        separator_x = (
            x_positions[last] + x_positions[first]
        ) / 2

        ax.axvline(
            separator_x,
            color="gray",
            linestyle="--",
            linewidth=1,
            alpha=0.7,
            zorder=0,
        )

def plot_teacher_chain(
    results,
    title="Learned-target teacher-chain diagnostic",
    filename="plot3_teacher_chain.png",
    model="Model B",
):
    teacher_sm = teacher_summary(results)

    if teacher_sm.empty:
        print("No learned-target teacher diagnostics are available.")
        return

    teacher_sm = teacher_sm[
        teacher_sm["model"] == model
    ].copy()

    if teacher_sm.empty:
        print(f"No teacher diagnostics for {model}.")
        return

    # ------------------------------------------------------------
    # Get GCN -> TRUE raw-space results separately
    # ------------------------------------------------------------
    gcn_sm = primary_summary(results)

    gcn_sm = gcn_sm[
        gcn_sm["model"] == model
    ].copy()

    # ------------------------------------------------------------
    # Match teacher-chain and GCN rows
    # ------------------------------------------------------------
    if "window" in teacher_sm.columns:
        merge_keys = [
            "model",
            "window",
            "target_modality",
        ]
    else:
        merge_keys = [
            "model",
            "target_modality",
        ]

    gcn_cols = merge_keys + [
        "cosine_mean",
        "cosine_ci_low",
        "cosine_ci_high",
    ]

    gcn_plot = gcn_sm[
        [c for c in gcn_cols if c in gcn_sm.columns]
    ].copy()

    gcn_plot = gcn_plot.rename(
        columns={
            "cosine_mean": "gcn_vs_true_raw_cosine_mean",
            "cosine_ci_low": "gcn_vs_true_raw_cosine_ci_low",
            "cosine_ci_high": "gcn_vs_true_raw_cosine_ci_high",
        }
    )

    sm = teacher_sm.merge(
        gcn_plot,
        on=merge_keys,
        how="left",
    )

    # ------------------------------------------------------------
    # Determine plotting order
    # ------------------------------------------------------------
    if "window" in sm.columns:

        keys = list(
            sm[
                ["window", "target_modality"]
            ]
            .drop_duplicates()
            .itertuples(
                index=False,
                name=None,
            )
        )

        labels = [
            target_modality
            for window, target_modality in keys
        ]

    else:

        keys = [
            (target_modality,)
            for target_modality
            in sm["target_modality"].drop_duplicates()
        ]

        labels = [
            target_modality
            for (target_modality,) in keys
        ]

    # ------------------------------------------------------------
    # Reorder rows to match x-axis
    # ------------------------------------------------------------
    ordered_rows = []

    for key in keys:

        if "window" in sm.columns:
            q = sm[
                (sm["window"] == key[0])
                & (sm["target_modality"] == key[1])
            ]
        else:
            q = sm[
                sm["target_modality"] == key[0]
            ]

        if not q.empty:
            ordered_rows.append(q.iloc[0])

    if not ordered_rows:
        print("No teacher-chain rows available.")
        return

    sm = pd.DataFrame(
        ordered_rows
    ).reset_index(drop=True)

    # Keep original group spacing to preserve window separator
    # x = np.arange(len(sm))
    group_spacing = 0.80
    x = np.arange(len(sm)) * group_spacing

    # ------------------------------------------------------------
    # Raw embedding-space cosine similarities
    # ------------------------------------------------------------
    links = [
        (
            "Learned target -> true",
            "teacher_vs_true_raw_cosine_mean",
            "teacher_vs_true_raw_cosine_ci_low",
            "teacher_vs_true_raw_cosine_ci_high",
        ),
        (
            "GCN -> learned target",
            "gcn_vs_teacher_raw_cosine_mean",
            "gcn_vs_teacher_raw_cosine_ci_low",
            "gcn_vs_teacher_raw_cosine_ci_high",
        ),
        (
            "GCN -> true",
            "gcn_vs_true_raw_cosine_mean",
            "gcn_vs_true_raw_cosine_ci_low",
            "gcn_vs_true_raw_cosine_ci_high",
        ),
    ]

    # ------------------------------------------------------------
    # Bar layout
    # ------------------------------------------------------------
    bar_width = 0.18
    bar_spacing = 0.21

    fig, ax = plt.subplots(
        figsize=(5.5, 3.4),
        dpi=600,
    )

    fig.patch.set_alpha(0.0)
    ax.patch.set_alpha(0.0)

    for li, (
        label,
        mean_col,
        lo_col,
        hi_col,
    ) in enumerate(links):

        for i, r in sm.iterrows():

            value = r.get(mean_col, np.nan)

            if pd.isna(value):
                continue

            p = float(value)

            lo = max(
                0.0,
                p - float(r[lo_col])
            )

            hi = max(
                0.0,
                float(r[hi_col]) - p
            )

            offset = (
                li - (len(links) - 1) / 2
            ) * bar_spacing

            bar_x = x[i] + offset

            ax.bar(
                bar_x,
                p,
                bar_width,
                color=CHAIN_COLORS[label],
                edgecolor="white",
                linewidth=0.5,
                yerr=[[lo], [hi]],
                capsize=2,
                error_kw=dict(
                    ecolor="lightgray",
                    elinewidth=1,
                ),
                alpha=0.9,
            )

            # Stagger labels to reduce overlapping values
            label_offsets = [0.012, 0.035, 0.012]

            ax.text(
                bar_x,
                p + hi + label_offsets[li],
                f"{p:.3f}",
                ha="center",
                va="bottom",
                fontsize=5.5,
                color="black",
            )

    # ------------------------------------------------------------
    # Current / dual window separator
    # ------------------------------------------------------------
    # if "window" in sm.columns:
    #     _add_window_separator_and_labels(
    #         ax,
    #         keys,
    #     )

    if "window" in sm.columns:
        _add_window_separator_and_labels_chain(
            ax,
            keys,
            x_positions=x,
            y_text=0.96,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        labels,
        fontsize=8,
    )

    ax.tick_params(
        axis="x",
        length=0,
    )

    ax.set_ylabel(
        "Raw cosine similarity"
    )

    ax.set_title(title)

    # ------------------------------------------------------------
    # Y-axis range
    # ------------------------------------------------------------
    available_values = []

    for _, mean_col, _, hi_col in links:

        if mean_col in sm.columns:
            available_values.extend(
                pd.to_numeric(
                    sm[mean_col],
                    errors="coerce",
                )
                .dropna()
                .tolist()
            )

        if hi_col in sm.columns:
            available_values.extend(
                pd.to_numeric(
                    sm[hi_col],
                    errors="coerce",
                )
                .dropna()
                .tolist()
            )

    if available_values:
        ax.set_ylim(
            0,
            min(
                1.08,
                max(available_values) + 0.12,
            ),
        )

    ax.legend(
        handles=[
            Patch(
                facecolor=CHAIN_COLORS[label],
                edgecolor="white",
                label=label,
            )
            for label, _, _, _ in links
        ],
        fontsize=8,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.20),
        frameon=False,
        ncol=3,
        handletextpad=0.6,
        borderaxespad=0.0,
    )

    plt.tight_layout()

    _savefig(fig, filename)

    plt.show()


In [ ]:
plot_primary_imputation(
    results,
    title='EXT-ST: GCN imputation quality',
    filename='hospitalisation_gcn_imputation_quality.png',
)

plot_paired_cl_effect(
    results,
    title='EXT-ST: change in GCN imputation quality with CL',
    filename='hospitalisation_cl_effect.png',
)

plot_teacher_chain(
    results,
    title='EXT-ST: learned-target teacher-chain diagnostic',
    filename='hospitalisation_teacher_chain.png',
    model='Model B',
)

## 5. Interpretation and reporting limitations

The figures assess **embedding reconstruction**, not downstream clinical prediction. Model-specific embedding spaces can differ, so raw cosine values are best interpreted within compatible evaluation settings. The paired CL effect requires valid encounter matching, especially when evaluation-order fallback IDs are used. Approximate normal-theory CIs do not automatically account for repeated admissions from the same patient. Learned-target alignment patterns are diagnostic associations, not causal proof of information transfer. Differences across windows may reflect cohort and context changes rather than CL alone.